# Tableau Metadata Bridge — Play 3

> **Part of the Tableau + Microsoft Fabric AI Bridge project.**

This notebook pulls governance metadata from the Tableau Metadata API (GraphQL) and lands
it in a Microsoft Fabric Lakehouse as four Delta tables:

| Table | Contents |
|-------|----------|
| `tableau_datasources` | One row per published data source — name, owner, certification, tags, connection type |
| `tableau_fields` | One row per field per data source — field name, type, description, calculated field formula |
| `tableau_lineage` | One row per relationship — upstream database tables and downstream workbooks per data source |
| `tableau_workbooks` | One row per workbook — name, owner, sheets, published datasource connections |

Together these tables enable a Fabric Data Agent to answer cross-platform governance questions like:
- *"Which database tables are used in both Tableau and Power BI?"*
- *"What data sources are certified and who owns them?"*
- *"Which workbooks would break if the Orders table changed?"*

## Steps
1. Configuration
2. Authenticate to Tableau
3. Query Metadata API (GraphQL) — datasources + workbooks
4. Parse datasources
5. Parse fields
6. Parse lineage
7. Parse workbooks
8. Write Delta tables to Lakehouse
9. Verify

> **Known limitation:** `certificationStatus` and `certifiedBy` fields require the
> Tableau Data Management add-on and are not included in this notebook. `isCertified`
> (boolean) is included. `workbook_owner` and `workbook_project` in the lineage table
> may appear as null on Tableau trial environments where workbooks are owned by
> 'Tableau System Account'.


## Variables to set before running

| Variable | What it is | Where to find it |
|----------|-----------|------------------|
| `PAT_NAME` | Tableau Personal Access Token name | Tableau → Account Settings → Personal Access Tokens |
| `POD` | Tableau Cloud pod hostname | First part of your Tableau Cloud URL e.g. `10ay.online.tableau.com` |
| `SITE` | Site contentUrl slug | Your site URL slug e.g. `mycompany`. Use `""` for Tableau Server default site |
| `KV_URL` | Azure Key Vault URL | portal.azure.com → your Key Vault → Overview → Vault URI |
| `KV_SECRET_NAME` | Name of the Key Vault secret storing your PAT secret | The secret name you used when storing the PAT in Key Vault |
| `DS_TABLE` | Delta table name for datasources | Your choice — e.g. `tableau_datasources` |
| `FIELDS_TABLE` | Delta table name for fields | Your choice — e.g. `tableau_fields` |
| `LINEAGE_TABLE` | Delta table name for lineage | Your choice — e.g. `tableau_lineage` |
| `WORKBOOKS_TABLE` | Delta table name for workbooks | Your choice — e.g. `tableau_workbooks` |


## Cell 1 — Configuration

Set your Tableau environment details here. The PAT secret is retrieved securely from
Azure Key Vault — no credentials are hardcoded in this notebook.

> 🔄 **Adapting for your environment:** `POD`, `SITE`, `KV_URL`, and `KV_SECRET_NAME`
> are the only values that change between environments. The table names are your choice.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
PAT_NAME     = ""          # PAT name from Tableau account settings
POD          = ""          # Tableau Cloud pod e.g. 10ay.online.tableau.com
                           # Tableau Server: use your server hostname
SITE         = ""          # Site contentUrl slug. Use "" for Tableau Server default site

KV_URL          = "https://<your-keyvault-name>.vault.azure.net/"
KV_SECRET_NAME  = "<your-secret-name>"

DS_TABLE      = "tableau_datasources"   # Delta table name for datasource metadata
FIELDS_TABLE  = "tableau_fields"        # Delta table name for field-level metadata
LINEAGE_TABLE = "tableau_lineage"       # Delta table name for lineage relationships
WORKBOOKS_TABLE = "tableau_workbooks"   # Delta table name for workbook metadata

# ── SECURE CREDENTIAL RETRIEVAL ───────────────────────────────────────────────
PAT_SECRET = notebookutils.credentials.getSecret(KV_URL, KV_SECRET_NAME)

BASE = f"https://{POD}"
print("✓ Configuration loaded")
print(f"  Pod:               {POD}")
print(f"  Site:              {SITE}")
print(f"  Datasources table: {DS_TABLE}")
print(f"  Fields table:      {FIELDS_TABLE}")
print(f"  Lineage table:     {LINEAGE_TABLE}")
print(f"  Workbooks table:   {WORKBOOKS_TABLE}")
print(f"  PAT secret:        retrieved from Key Vault ✓")

## Cell 2 — Authenticate to Tableau

Authenticates using your PAT and retrieves a session token. The Metadata API uses the
same PAT authentication as the REST API and VDS.

> **If you get a 401 error in later cells**, re-run this cell to refresh the session token.
> Tokens expire after inactivity or if another session opens with the same PAT.

In [ ]:
import requests
import json
import pandas as pd
from datetime import datetime

auth_url = f"{BASE}/api/3.24/auth/signin"
auth_payload = {
    "credentials": {
        "personalAccessTokenName": PAT_NAME,
        "personalAccessTokenSecret": PAT_SECRET,
        "site": {"contentUrl": SITE}
    }
}

auth_response = requests.post(auth_url, json=auth_payload, headers={"Accept": "application/json"})
auth_response.raise_for_status()

auth_data   = auth_response.json()
TOKEN       = auth_data["credentials"]["token"]
SITE_ID     = auth_data["credentials"]["site"]["id"]

METADATA_URL = f"{BASE}/api/metadata/graphql"
HEADERS = {
    "X-Tableau-Auth": TOKEN,
    "Content-Type": "application/json",
    "Accept": "application/json"
}

print("✓ Authenticated to Tableau")
print(f"  Site ID: {SITE_ID}")
print(f"  Token:   {TOKEN[:8]}...")

## Cell 3 — Query the Tableau Metadata API

Sends a single GraphQL query that pulls all published data sources with their:
- Name, description, owner, project, certification status, tags
- All fields (including calculated field formulas)
- Upstream database tables (lineage)
- Downstream workbooks (lineage)

The Metadata API uses GraphQL — one query, one endpoint, exactly the fields we ask for.
Pagination is handled automatically using cursor-based pagination (1000 items per page).

> 🔄 **Adapting for your environment:** The query fields are fixed — they pull everything
> the Metadata API exposes for published data sources. No changes needed here unless you
> want to add or remove specific fields from the output tables.

In [ ]:
METADATA_QUERY = """
query GetPublishedDatasources($cursor: String) {
  publishedDatasourcesConnection(first: 100, after: $cursor) {
    pageInfo {
      hasNextPage
      endCursor
    }
    nodes {
      id
      name
      description
      projectName
      hasExtracts
      extractLastRefreshTime
      isCertified
      tags { name }
      owner { name username }
      upstreamDatabases { name connectionType }
      upstreamTables {
        name
        schema
        fullName
        database { name connectionType }
      }
      downstreamWorkbooks {
        name
        projectName
        owner { name }
      }
      fields {
        id
        name
        description
        isHidden
        __typename
        ... on ColumnField {
          dataType
          role
        }
        ... on CalculatedField {
          dataType
          role
          formula
        }
        ... on GroupField {
          dataType
          role
        }
        ... on HierarchyField {
          __typename
        }
        ... on SetField {
          __typename
        }
      }
    }
  }
}
"""
WORKBOOKS_QUERY = """
query GetWorkbooks($cursor: String) {
  workbooksConnection(first: 100, after: $cursor) {
    pageInfo {
      hasNextPage
      endCursor
    }
    nodes {
      id
      name
      description
      projectName
      createdAt
      updatedAt
      tags { name }
      owner { name username }
      sheets {
        name
      }
      upstreamDatasources {
        name
        id
      }
      embeddedDatasources {
        name
      }
    }
  }
}
"""

# ── PAGINATED FETCH — WORKBOOKS ────────────────────────────────────────────────
all_workbooks = []
cursor = None
page = 1

while True:
    variables = {"cursor": cursor}
    response = requests.post(
        METADATA_URL,
        headers=HEADERS,
        json={"query": WORKBOOKS_QUERY, "variables": variables}
    )
    response.raise_for_status()
    data = response.json()

    if "errors" in data:
        raise Exception(f"GraphQL error: {data['errors']}")

    connection = data["data"]["workbooksConnection"]
    nodes      = connection["nodes"]
    page_info  = connection["pageInfo"]

    all_workbooks.extend(nodes)
    print(f"  Workbooks page {page}: {len(nodes)} fetched (total: {len(all_workbooks)})")

    if not page_info["hasNextPage"]:
        break

    cursor = page_info["endCursor"]
    page += 1

print(f"\n✓ Workbook query complete")
print(f"  Total workbooks: {len(all_workbooks)}")
# ── PAGINATED FETCH ────────────────────────────────────────────────────────────
all_datasources = []
cursor = None
page = 1

while True:
    variables = {"cursor": cursor}
    response = requests.post(
        METADATA_URL,
        headers=HEADERS,
        json={"query": METADATA_QUERY, "variables": variables}
    )
    response.raise_for_status()
    data = response.json()

    if "errors" in data:
        raise Exception(f"GraphQL error: {data['errors']}")

    connection = data["data"]["publishedDatasourcesConnection"]
    nodes      = connection["nodes"]
    page_info  = connection["pageInfo"]

    all_datasources.extend(nodes)
    print(f"  Page {page}: {len(nodes)} datasources fetched (total so far: {len(all_datasources)})")

    if not page_info["hasNextPage"]:
        break

    cursor = page_info["endCursor"]
    page += 1

print(f"\n✓ Metadata query complete")
print(f"  Total published data sources: {len(all_datasources)}")

## Cell 4 — Parse Datasource Metadata

Builds the `tableau_datasources` table — one row per published data source.
Captures name, owner, project, certification, extract status, connection types, and tags.

This is the top-level catalog view — the equivalent of what you'd see browsing
Tableau's data source list with governance overlaid.

In [ ]:
EXTRACTED_AT = datetime.utcnow().isoformat()

ds_rows = []
for ds in all_datasources:
    ds_rows.append({
        "datasource_id":           ds.get("id"),
        "name":                    ds.get("name"),
        "description":             ds.get("description"),
        "project_name":            ds.get("projectName"),
        "owner_name":              ds.get("owner", {}).get("name") if ds.get("owner") else None,
        "owner_username":          ds.get("owner", {}).get("username") if ds.get("owner") else None,
        "is_certified":            ds.get("isCertified", False),
        "certification_status":    ds.get("certificationStatus"),
        "certified_by":            ds.get("certifiedBy"),
        "has_extracts":            ds.get("hasExtracts", False),
        "extract_last_refresh":    ds.get("extractLastRefreshTime"),
        "upstream_databases":      ", ".join([db.get("name", "") for db in ds.get("upstreamDatabases", [])]),
        "connection_types":        ", ".join(list(set([db.get("connectionType", "") for db in ds.get("upstreamDatabases", [])]))),
        "tags":                    ", ".join([t.get("name", "") for t in ds.get("tags", [])]),
        "downstream_workbook_count": len(ds.get("downstreamWorkbooks", [])),
        "field_count":             len(ds.get("fields", [])),
        "extracted_at":            EXTRACTED_AT
    })

df_datasources = pd.DataFrame(ds_rows)
print(f"✓ Datasources parsed: {len(df_datasources)} rows")
print(f"  Columns: {list(df_datasources.columns)}")
df_datasources.head()

## Cell 7 — Parse Workbook Metadata

Builds the `tableau_workbooks` table — one row per workbook.
Captures name, owner, project, sheet count, sheet names, and
which published data sources each workbook connects to.

This is the consumption layer — shows who is using which data sources
and how many sheets each workbook contains.

In [ ]:
# ── PARSE WORKBOOKS ────────────────────────────────────────────────────────────
wb_rows = []
for wb in all_workbooks:
    wb_rows.append({
        "workbook_id":              wb.get("id"),
        "name":                     wb.get("name"),
        "description":              wb.get("description"),
        "project_name":             wb.get("projectName"),
        "owner_name":               wb.get("owner", {}).get("name") if wb.get("owner") else None,
        "owner_username":           wb.get("owner", {}).get("username") if wb.get("owner") else None,
        "created_at":               wb.get("createdAt"),
        "updated_at":               wb.get("updatedAt"),
        "tags":                     ", ".join([t.get("name", "") for t in wb.get("tags", [])]),
        "sheet_count":              len(wb.get("sheets", [])),
        "sheet_names":              ", ".join([s.get("name", "") for s in wb.get("sheets", [])]),
        "published_datasources":    ", ".join([d.get("name", "") for d in wb.get("upstreamDatasources", [])]),
        "embedded_datasources":     ", ".join([d.get("name", "") for d in wb.get("embeddedDatasources", [])]),
        "published_datasource_count": len(wb.get("upstreamDatasources", [])),
        "extracted_at":             EXTRACTED_AT
    })

df_workbooks = pd.DataFrame(wb_rows)
print(f"✓ Workbooks parsed: {len(df_workbooks)} rows")
print(f"  Columns: {list(df_workbooks.columns)}")
df_workbooks.head()

## Cell 5 — Parse Field Metadata

Builds the `tableau_fields` table — one row per field per data source.
Captures field name, type, role (dimension/measure), description, and formula
for calculated fields.

This is the schema/data dictionary layer — useful for understanding what's in each
data source without opening Tableau.

In [ ]:
field_rows = []
for ds in all_datasources:
    ds_id   = ds.get("id")
    ds_name = ds.get("name")
    for field in ds.get("fields", []):
        field_rows.append({
            "datasource_id":   ds_id,
            "datasource_name": ds_name,
            "field_id":        field.get("id"),
            "field_name":      field.get("name"),
            "field_type":      field.get("__typename"),   # ColumnField, CalculatedField, GroupField, etc.
            "data_type":       field.get("dataType"),     # STRING, INTEGER, REAL, BOOLEAN, DATE, DATETIME
            "role":            field.get("role"),          # DIMENSION or MEASURE
            "description":     field.get("description"),
            "formula":         field.get("formula"),       # Only populated for CalculatedField
            "is_hidden":       field.get("isHidden", False),
            "extracted_at":    EXTRACTED_AT
        })

df_fields = pd.DataFrame(field_rows)
print(f"✓ Fields parsed: {len(df_fields)} rows")
print(f"  Field types found: {df_fields['field_type'].value_counts().to_dict()}")
df_fields.head()

## Cell 6 — Parse Lineage

Builds the `tableau_lineage` table — one row per relationship.
Two relationship types are captured:

- `upstream_table` — the database table that feeds this data source
- `downstream_workbook` — the workbook that consumes this data source

This is the cross-platform lineage layer. Combined with Power BI semantic model lineage
(Phase 2 of Play 3), it enables queries like:
*"Which upstream database tables are shared between Tableau and Power BI?"*

> 🔄 **Adapting for your environment:** No changes needed. The lineage relationships
> are pulled directly from Tableau's metadata model.

In [ ]:
lineage_rows = []
for ds in all_datasources:
    ds_id   = ds.get("id")
    ds_name = ds.get("name")

    # Upstream: database tables that feed this datasource
    for table in ds.get("upstreamTables", []):
        lineage_rows.append({
            "datasource_id":        ds_id,
            "datasource_name":      ds_name,
            "relationship_type":    "upstream_table",
            "related_asset_name":   table.get("name"),
            "related_asset_schema": table.get("schema"),
            "related_asset_full":   table.get("fullName"),
            "database_name":        table.get("database", {}).get("name") if table.get("database") else None,
            "connection_type":      table.get("database", {}).get("connectionType") if table.get("database") else None,
            "workbook_owner":       None,
            "workbook_project":     None,
            "extracted_at":         EXTRACTED_AT
        })

    # Downstream: workbooks that consume this datasource
    for wb in ds.get("downstreamWorkbooks", []):
        lineage_rows.append({
            "datasource_id":        ds_id,
            "datasource_name":      ds_name,
            "relationship_type":    "downstream_workbook",
            "related_asset_name":   wb.get("name"),
            "related_asset_schema": None,
            "related_asset_full":   None,
            "database_name":        None,
            "connection_type":      None,
            "workbook_owner":       wb.get("owner", {}).get("name") if wb.get("owner") else None,
            "workbook_project":     wb.get("projectName"),
            "extracted_at":         EXTRACTED_AT
        })

df_lineage = pd.DataFrame(lineage_rows)
print(f"✓ Lineage parsed: {len(df_lineage)} rows")
print(f"  Relationship types: {df_lineage['relationship_type'].value_counts().to_dict()}")
df_lineage.head()

## Cell 7 — Write Delta Tables to Lakehouse

Writes all three DataFrames to the attached Lakehouse as Delta tables.
Uses `overwrite` mode so each run produces a fresh snapshot of the current
Tableau metadata state.

> 🔄 **Adapting for your environment:** Make sure a Lakehouse is attached to this
> notebook in the Explorer pane (left rail) before running. The table names are
> controlled by `DS_TABLE`, `FIELDS_TABLE`, and `LINEAGE_TABLE` in Cell 1.

In [ ]:
def write_delta(df_pandas, table_name):
    """Convert pandas DataFrame to Spark and write as Delta table."""
    df_spark = spark.createDataFrame(df_pandas)
    (
        df_spark.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    count = spark.table(table_name).count()
    print(f"  ✓ {table_name}: {count} rows written")
    return count

print("Writing Delta tables...")
write_delta(df_datasources, DS_TABLE)
write_delta(df_fields,      FIELDS_TABLE)
write_delta(df_lineage,     LINEAGE_TABLE)
write_delta(df_workbooks, "tableau_workbooks")
print(f"\n✓ All four tables written at {EXTRACTED_AT}")

## Cell 8 — Verify

Runs a few spot-check queries to confirm the data landed correctly and looks reasonable.
Also previews the kind of cross-datasource questions the tables can answer.

In [ ]:
print("=" * 60)
print("VERIFICATION")
print("=" * 60)

# Row counts
print("\n── Row counts ──")
for table in [DS_TABLE, FIELDS_TABLE, LINEAGE_TABLE, WORKBOOKS_TABLE]:
    count = spark.table(table).count()
    print(f"  {table}: {count} rows")

# Certified datasources
print("\n── Certified data sources ──")
spark.sql(f"""
    SELECT name, owner_name, project_name
    FROM {DS_TABLE}
    WHERE is_certified = true
    ORDER BY name
""").show(truncate=False)

# Calculated fields across all datasources
print("── Calculated fields ──")
spark.sql(f"""
    SELECT datasource_name, field_name, formula
    FROM {FIELDS_TABLE}
    WHERE field_type = 'CalculatedField'
    AND formula IS NOT NULL
    ORDER BY datasource_name, field_name
    LIMIT 10
""").show(truncate=False)

# Most-used upstream tables (which database tables are queried most by Tableau)
print("── Most-used upstream database tables ──")
spark.sql(f"""
    SELECT database_name, related_asset_name, connection_type,
           COUNT(*) as datasource_count
    FROM {LINEAGE_TABLE}
    WHERE relationship_type = 'upstream_table'
    AND database_name IS NOT NULL
    GROUP BY database_name, related_asset_name, connection_type
    ORDER BY datasource_count DESC
    LIMIT 10
""").show(truncate=False)

# Workbooks with most datasource dependencies
print("── Workbooks with most Tableau datasource dependencies ──")
spark.sql(f"""
    SELECT related_asset_name as workbook_name,
           COUNT(*) as datasource_count
    FROM {LINEAGE_TABLE}
    WHERE relationship_type = 'downstream_workbook'
    GROUP BY related_asset_name
    ORDER BY datasource_count DESC
    LIMIT 10
""").show(truncate=False)

print("\n✓ Verification complete")
print(f"  Tables ready for Fabric Data Agent: {DS_TABLE}, {FIELDS_TABLE}, {LINEAGE_TABLE}, {WORKBOOKS_TABLE}")

# Workbooks and their published datasource dependencies
print("── Workbooks and their published datasource connections ──")
spark.sql("""
    SELECT name, project_name, owner_name, 
           sheet_count, published_datasources
    FROM tableau_workbooks
    ORDER BY name
""").show(truncate=False)